In [1]:
'''
task: FOL classification based on graph data from AST
model: graphormer
dataset: P-FOLIO
evaluation: supervised fine-tuning
'''
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# load FOLIO dataset
import pandas as pd

fol_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/pfolio_ast_train.csv")
fol_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/pfolio_ast_test.csv")
fol_valid = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/pfolio_ast_val.csv")

In [3]:
# logic to generate graph from AST
import json
import matplotlib.pyplot as plt
import networkx as nx

def extract_text(node):
    """Recursively processes specific sub-tokens to rebuild logic strings."""
    pieces = []
    if "text" in node:
        pieces.append(node["text"])
    if "children" in node:
        for child in node["children"]:
            pieces.extend(extract_text(child))
    return "".join(pieces)

def clean_syntax(text):
    """Formats string operations into formal math notation markers."""
    return (text.strip()
            .replace("forall", "∀ ")
            .replace("implies", " → ")
            .replace("and", " ∧ ")
            .replace("not", " ¬ ")
            .replace("()", ""))

def ast_to_syllogism_graph(ast_dict):
  """Transforms an AST into a directed NetworkX graph where all nodes

  sharing the same name/label (regardless of type, e.g., symbol 'x' and term 'x')
  are merged into a single unique node, accumulating all incoming and
  outgoing edges.
  """
  G = nx.DiGraph()

  # 1. Add Root Node
  root_node = "Syllogism"
  G.add_node(root_node, label="Syllogism", node_type="root")

  # Helper to safely locate statements in the AST structure
  def find_statements(node):
    stats = []
    if not isinstance(node, dict):
      return stats

    if node.get("type") == "stat":
      stats.append(node)

    for child in node.get("children", []):
      stats.extend(find_statements(child))

    return stats

  statements = find_statements(ast_dict)
  if not statements:
    return G

  premises = statements[:-1]
  conclusion = statements[-1]

  # Helper to split a term node's contents by commas (e.g., x,y or tecmo,japan)
  def extract_sub_terms_from_node(node):
    if not isinstance(node, dict):
      return []

    sub_terms = []
    current_letters = []
    for child in node.get("children", []):
      child_type = child.get("type")
      if child_type == "COMMA":
        if current_letters:
          sub_terms.append("".join(current_letters))
          current_letters = []
      elif child_type == "LETTER" and "text" in child:
        current_letters.append(child["text"])
      else:
        nested = extract_sub_terms_from_node(child)
        if nested:
          sub_terms.extend(nested)
    if current_letters:
      sub_terms.append("".join(current_letters))

    if not sub_terms and "text" in node:
      sub_terms.append(node["text"])

    return sub_terms

  # Helper to find text inside simple nodes like quantifier, keyword, symbol
  def get_node_text(node):
    if not isinstance(node, dict):
      return ""
    if "text" in node:
      return node["text"]
    for child in node.get("children", []):
      res = get_node_text(child)
      if res:
        return res
    return ""

  def collect_keywords(node, keywords):
    if not isinstance(node, dict):
      return
    if node.get("type") == "keyword":
      kw_text = get_node_text(node)
      if kw_text and kw_text not in keywords:
        keywords.append(kw_text)
    for child in node.get("children", []):
      collect_keywords(child, keywords)

  def find_atomic_propositions(node):
    aps = []
    if not isinstance(node, dict):
      return aps
    if node.get("type") == "atomicproposition":
      aps.append(node)
    for child in node.get("children", []):
      aps.extend(find_atomic_propositions(child))
    return aps

  # Helper to ensure unique nodes by label name (merges different types with same name)
  def add_merged_node(label, node_type):
    if not G.has_node(label):
      G.add_node(label, label=label, node_type=node_type)
    else:
      # Optional: update node_type to reflect multiple roles if needed
      existing_type = G.nodes[label].get("node_type", "")
      if node_type not in existing_type:
        G.nodes[label]["node_type"] = f"{existing_type},{node_type}"
    return label

  # Process a single statement (Premise or Conclusion)
  def process_statement(stat_node, parent_graph_node):
    quantifier_label = None
    symbol_label = None

    def find_quantifier_symbol(node):
      nonlocal quantifier_label, symbol_label
      if not isinstance(node, dict):
        return
      ntype = node.get("type")
      if ntype == "quantifier":
        q_text = get_node_text(node)
        if q_text:
          quantifier_label = add_merged_node(q_text, "quantifier")
          G.add_edge(parent_graph_node, quantifier_label)
      elif ntype == "symbol" and quantifier_label:
        s_text = get_node_text(node)
        if s_text:
          symbol_label = add_merged_node(s_text, "symbol")
          G.add_edge(quantifier_label, symbol_label)

      for child in node.get("children", []):
        find_quantifier_symbol(child)

    find_quantifier_symbol(stat_node)

    # Process Atomic Propositions (Predicate -> Argument Term Edges)
    aps = find_atomic_propositions(stat_node)
    statement_terms = set()

    for ap in aps:
      term_children = [
          c for c in ap.get("children", []) if c.get("type") == "term"
      ]
      ap_terms = []
      for tc in term_children:
        sub = extract_sub_terms_from_node(tc)
        ap_terms.extend(sub)

      if ap_terms:
        predicate_text = ap_terms[0]
        pred_label = add_merged_node(predicate_text, "term")
        G.add_edge(parent_graph_node, pred_label)
        statement_terms.add(predicate_text)

        for arg_text in ap_terms[1:]:
          arg_label = add_merged_node(arg_text, "term")
          G.add_edge(parent_graph_node, arg_label)
          statement_terms.add(arg_text)

          # Directed edge from predicate to argument term
          G.add_edge(pred_label, arg_label)

    # Fallback for any standalone terms not caught in atomic propositions
    def find_all_terms(node):
      if not isinstance(node, dict):
        return
      if node.get("type") == "term":
        for t in extract_sub_terms_from_node(node):
          if t not in statement_terms:
            t_label = add_merged_node(t, "term")
            G.add_edge(parent_graph_node, t_label)
            statement_terms.add(t)
      for child in node.get("children", []):
        find_all_terms(child)

    find_all_terms(stat_node)

    # Keywords -> Connect keyword to all terms in the statement
    keywords = []
    collect_keywords(stat_node, keywords)

    for kw in keywords:
      kw_label = add_merged_node(kw, "keyword")
      G.add_edge(parent_graph_node, kw_label)

      # Connect keyword to all terms present in this statement
      for t in statement_terms:
        G.add_edge(kw_label, t)

  # 2. Process Premises
  for idx, premise in enumerate(premises):
    premise_name = f"Premise_{idx + 1}"
    add_merged_node(premise_name, "premise")
    G.add_edge(root_node, premise_name)
    process_statement(premise, premise_name)

  # 3. Process Conclusion
  conclusion_name = "Conclusion"
  add_merged_node(conclusion_name, "conclusion")
  G.add_edge(root_node, conclusion_name)
  process_statement(conclusion, conclusion_name)

  return G

In [4]:
# generate graphs from PFOLIO train dataset
train_graph_list = [] # graph transformation of each syllogism
train_label_list = [] # label for each graph (syllogism)
for idx, row in fol_train.iterrows():
    premises_ast = row["Premises - CLIF"]
    conclusion_ast = row["Conclusions - CLIF"]
    # replace single quotes with double quotes to make it valid JSON
    premises_ast = premises_ast.replace("'", '"')
    conclusion_ast = conclusion_ast.replace("'", '"')
    # add conlusion as a child to premises dict
    premises_ast = json.loads(premises_ast)
    conclusion_ast = json.loads(conclusion_ast)
    premises_ast["children"].append(conclusion_ast["children"][0]["children"][0])
    premises_ast = premises_ast["children"][0]  # get the first child of premises as the root of the AST
    ast_data = premises_ast
    # generate graph from AST
    ast_graph = ast_to_syllogism_graph(ast_data)
    train_graph_list.append(ast_graph)
    train_label_list.append(row["Truth Values"])

print("*** Train data graph dataset generation complete ***")

# generate graphs from PFOLIO test dataset
test_graph_list = [] # graph transformation of each syllogism
test_label_list = [] # label for each graph (syllogism)
for idx, row in fol_test.iterrows():
    premises_ast = row["Premises - CLIF"]
    conclusion_ast = row["Conclusions - CLIF"]
    # replace single quotes with double quotes to make it valid JSON
    premises_ast = premises_ast.replace("'", '"')
    conclusion_ast = conclusion_ast.replace("'", '"')
    # add conlusion as a child to premises dict
    premises_ast = json.loads(premises_ast)
    conclusion_ast = json.loads(conclusion_ast)
    premises_ast["children"].append(conclusion_ast["children"][0]["children"][0])
    premises_ast = premises_ast["children"][0]  # get the first child of premises as the root of the AST
    ast_data = premises_ast
    # generate graph from AST
    ast_graph = ast_to_syllogism_graph(ast_data)
    test_graph_list.append(ast_graph)
    test_label_list.append(row["Truth Values"])

print("*** Test data graph dataset generation complete ***")

# generate graphs from PFOLIO val dataset
val_graph_list = [] # graph transformation of each syllogism
val_label_list = [] # label for each graph (syllogism)
for idx, row in fol_valid.iterrows():
    premises_ast = row["Premises - CLIF"]
    conclusion_ast = row["Conclusions - CLIF"]
    # replace single quotes with double quotes to make it valid JSON
    premises_ast = premises_ast.replace("'", '"')
    conclusion_ast = conclusion_ast.replace("'", '"')
    # add conlusion as a child to premises dict
    premises_ast = json.loads(premises_ast)
    conclusion_ast = json.loads(conclusion_ast)
    premises_ast["children"].append(conclusion_ast["children"][0]["children"][0])
    premises_ast = premises_ast["children"][0]  # get the first child of premises as the root of the AST
    ast_data = premises_ast
    # generate graph from AST
    ast_graph = ast_to_syllogism_graph(ast_data)
    val_graph_list.append(ast_graph)
    val_label_list.append(row["Truth Values"])

print("*** Val data graph dataset generation complete ***")

*** Train data graph dataset generation complete ***
*** Test data graph dataset generation complete ***
*** Val data graph dataset generation complete ***


In [5]:
!pip install scikit-learn torch-geometric
!pip install tensorboard

In [6]:
import torch
import networkx as nx
from torch_geometric.data import Data

def prepare_dataset_from_graphs(graphs_list, labels_list):
    """
    Converts a list of independent NetworkX DiGraphs into a list of PyTorch Geometric Data objects.

    Parameters:
    -----------
    graphs_list : list of nx.DiGraph
        A list where each element is a separate syllogism DiGraph (G).
    labels_list : list of int
        A list of integer labels corresponding to each graph in graphs_list.
        Values must be 0 (False), 1 (True), or 2 (Unknown).

    Returns:
    --------
    pyg_data_list : list of torch_geometric.data.Data
        A list of data objects ready for the PyTorch Geometric DataLoader.
    """
    # Map textual node types to unique integer categorical IDs
    type_map = {
        "root": 0,
        "premise": 1,
        "conclusion": 2,
        "quantifier": 3,
        "operator": 4,
        "proposition": 5
    }

    pyg_data_list = []

    # Iterate through every graph-label pair in your dataset
    for idx, G in enumerate(graphs_list):
        node_list = list(G.nodes())
        node_to_idx = {node: i for i, node in enumerate(node_list)}

        # 1. Extract and map node category features
        node_features = []
        for node in node_list:
            if node == "SYLLOGISM":
                n_type = "root"
            elif node == "C":
                n_type = "conclusion"
            elif node.startswith("P") and "_" not in node:
                n_type = "premise"
            elif "_q_" in node:
                n_type = "quantifier"
            elif "_op_" in node:
                n_type = "operator"
            else:
                n_type = "proposition"

            node_features.append(type_map[n_type])

        # Convert node features to a long tensor shape [num_nodes, 1]
        x = torch.tensor(node_features, dtype=torch.long).view(-1, 1)

        # 2. Extract edge combinations safely if edges exist
        edges = list(G.edges())
        if len(edges) > 0:
            edge_index = torch.tensor(
                [[node_to_idx[u], node_to_idx[v]] for u, v in edges],
                dtype=torch.long
            ).t().contiguous()
        else:
            # Handle empty edge fallback edge index shape [2, 0]
            edge_index = torch.empty((2, 0), dtype=torch.long)

        # 3. Associate the structural classification target label
        y = torch.tensor([labels_list[idx]], dtype=torch.long)

        # 4. Enforce structural grouping into PyG Data format
        data_object = Data(x=x, edge_index=edge_index, y=y)
        pyg_data_list.append(data_object)

    return pyg_data_list


In [7]:
!pip install -U transformers==4.45

In [14]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader
from transformers import GraphormerConfig, GraphormerForGraphClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split

# ==========================================
# 1. EARLY STOPPING FOR TRANSFORMERS
# ==========================================
class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.0005, checkpoint_path="best_graphormer_model.pt"):
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"EarlyStopping Counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.checkpoint_path)

# ==========================================
# 2. DATASET & BATCH COLLATION
# ==========================================
class GraphormerSyllogismDataset(Dataset):
    def __init__(self, graphs_list, labels_list, max_nodes=32):
        super().__init__()
        self.samples = []
        type_map = {"root": 0, "premise": 1, "conclusion": 2, "quantifier": 3, "operator": 4, "proposition": 5}

        for idx, G in enumerate(graphs_list):
            node_list = list(G.nodes())
            num_nodes = len(node_list)

            # Extract degree arrays
            in_deg_dict = dict(G.in_degree())
            out_deg_dict = dict(G.out_degree())

            node_features = []
            for node in node_list:
                if node == "SYLLOGISM": n_type = "root"
                elif node == "C": n_type = "conclusion"
                elif node.startswith("P") and "_" not in node: n_type = "premise"
                elif "_q_" in node: n_type = "quantifier"
                elif "_op_" in node: n_type = "operator"
                else: n_type = "proposition"

                # Concatenate node type categorical ID, in-degree, and out-degree
                type_id = type_map[n_type]
                in_d = in_deg_dict.get(node, 0)
                out_d = out_deg_dict.get(node, 0)
                node_features.append([type_id, in_d, out_d])

            input_nodes = torch.tensor(node_features, dtype=torch.long) # shape [num_nodes, 3]

            edge_attr = np.zeros((num_nodes, num_nodes, 1), dtype=np.int64)
            for u, v in G.edges():
                u_idx, v_idx = node_list.index(u), node_list.index(v)
                edge_attr[u_idx, v_idx, 0] = 1
            edge_input = torch.tensor(edge_attr, dtype=torch.long)

            shortest_paths = nx.all_pairs_shortest_path_length(G.to_undirected())
            spatial_pos_matrix = np.zeros((num_nodes, num_nodes), dtype=np.int64)
            path_dict = dict(shortest_paths)

            for i, u in enumerate(node_list):
                for j, v in enumerate(node_list):
                    spatial_pos_matrix[i, j] = path_dict[u][v] if v in path_dict[u] else max_nodes
            spatial_pos = torch.tensor(spatial_pos_matrix, dtype=torch.long)

            in_degree = torch.tensor([d for _, d in G.in_degree()], dtype=torch.long)
            out_degree = torch.tensor([d for _, d in G.out_degree()], dtype=torch.long)
            y = torch.tensor(labels_list[idx], dtype=torch.long)

            # Graphormer additional features required by forward()
            attn_bias = torch.zeros(num_nodes + 1, num_nodes + 1, dtype=torch.float)
            input_edges = torch.zeros(num_nodes, num_nodes, 1, 1, dtype=torch.long)
            attn_edge_type = torch.zeros(num_nodes, num_nodes, 1, dtype=torch.long)

            self.samples.append({
                "input_nodes": input_nodes, "edge_input": edge_input,
                "spatial_pos": spatial_pos, "in_degree": in_degree,
                "out_degree": out_degree, "labels": y,
                "attn_bias": attn_bias, "input_edges": input_edges,
                "attn_edge_type": attn_edge_type
            })

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def graphormer_collate_fn(batch):
    max_batch_nodes = max(item["input_nodes"].size(0) for item in batch)
    batch_input_nodes, batch_edge_input, batch_spatial_pos = [], [], []
    batch_in_degree, batch_out_degree, batch_labels = [], [], []
    batch_attn_bias, batch_input_edges, batch_attn_edge_type = [], [], []

    for item in batch:
        n = item["input_nodes"].size(0)
        pad_size = max_batch_nodes - n

        # Padding input_nodes of shape [n, 3] to [max_batch_nodes, 3]
        batch_input_nodes.append(torch.cat([item["input_nodes"], torch.zeros((pad_size, 3), dtype=torch.long)], dim=0))
        batch_edge_input.append(F.pad(item["edge_input"], (0, 0, 0, pad_size, 0, pad_size), value=0))
        batch_spatial_pos.append(F.pad(item["spatial_pos"], (0, pad_size, 0, pad_size), value=32))
        batch_in_degree.append(torch.cat([item["in_degree"], torch.zeros(pad_size, dtype=torch.long)], dim=0))
        batch_out_degree.append(torch.cat([item["out_degree"], torch.zeros(pad_size, dtype=torch.long)], dim=0))
        batch_labels.append(item["labels"])

        # Padding for attention matrices: use -1e9 instead of -inf to prevent Softmax 0/0 NaN division errors
        padded_attn_bias = F.pad(item["attn_bias"], (0, pad_size, 0, pad_size), value=-1e9)
        padded_attn_bias[0, 0] = 0.0
        batch_attn_bias.append(padded_attn_bias)
        batch_input_edges.append(F.pad(item["input_edges"], (0, 0, 0, 0, 0, pad_size, 0, pad_size), value=0))
        batch_attn_edge_type.append(F.pad(item["attn_edge_type"], (0, 0, 0, pad_size, 0, pad_size), value=0))

    return {
        "input_nodes": torch.stack(batch_input_nodes), "edge_input": torch.stack(batch_edge_input),
        "spatial_pos": torch.stack(batch_spatial_pos), "in_degree": torch.stack(batch_in_degree),
        "out_degree": torch.stack(batch_out_degree), "labels": torch.stack(batch_labels),
        "attn_bias": torch.stack(batch_attn_bias), "input_edges": torch.stack(batch_input_edges),
        "attn_edge_type": torch.stack(batch_attn_edge_type)
    }

# ==========================================
# 3. PIPELINE ORCHESTRATION
# ==========================================
def run_graphormer_pipeline(train_graph_list, train_label_list, test_graph_list, test_label_list, val_graph_list, val_label_list):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing Graphormer Pipeline on: {device}\n")

    writer = SummaryWriter(log_dir="runs/graphormer_classification")
    early_stopping = EarlyStopping(patience=8, checkpoint_path="best_graphormer_model.pt")

    # Initialize label mapping dictionary: ['0: F', '1: T', '2: U']
    label_map = {'F': 0, 'T': 1, 'U': 2}
    train_labels = [label_map[lbl] for lbl in train_label_list]
    test_labels = [label_map[lbl] for lbl in test_label_list]
    val_labels = [label_map[lbl] for lbl in val_label_list]

    # Construct dataset and standard PyTorch loader using our custom collate function
    train_dataset = GraphormerSyllogismDataset(graphs_list=train_graph_list, labels_list=train_labels, max_nodes=32)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=graphormer_collate_fn)

    test_dataset = GraphormerSyllogismDataset(graphs_list=test_graph_list, labels_list=test_labels, max_nodes=32)
    test_loader = DataLoader(test_dataset, batch_size=8, collate_fn=graphormer_collate_fn)

    val_dataset = GraphormerSyllogismDataset(graphs_list=val_graph_list, labels_list=val_labels, max_nodes=32)
    val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=graphormer_collate_fn)

    # Imbalance Class weights calculation
    class_counts = np.bincount(train_labels, minlength=3)
    class_weights = 1.0 / (class_counts + 1e-6)
    weight_tensor = torch.tensor(class_weights / np.sum(class_weights) * 3.0, dtype=torch.float).to(device)

    # Optimized configuration: num_atoms=64 to cover node types as well as high-degree mapping indices
    config = GraphormerConfig(
        num_classes=3, num_atoms=64, num_edges=2, spatial_pos_max=64,
        num_attention_heads=4, embedding_dim=64, ffn_embedding_dim=64, num_trans_layers=3,
        dropout=0.12, attention_dropout=0.08
    )
    model = GraphormerForGraphClassification(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.1)

    # Introduce label smoothing to make predictions less polarized
    criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=0.05)

    # Learning rate decay scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)

    num_epochs = 40
    print("--- Beginning Graphormer Training Loop ---")

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_train_loss, correct_train, total_graphs = 0, 0, 0

        for batch in train_loader:
            targets = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = criterion(outputs.logits, targets)
            loss.backward()

            # Apply gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            total_train_loss += loss.item() * targets.size(0)
            correct_train += int((outputs.logits.argmax(dim=-1) == targets).sum())
            total_graphs += targets.size(0)

        scheduler.step()
        avg_train_loss = total_train_loss / total_graphs
        train_acc = correct_train / total_graphs

        # Validation Loop
        model.eval()
        total_val_loss, correct_val, total_val_graphs = 0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                targets = batch.pop("labels").to(device)
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = criterion(outputs.logits, targets)

                total_val_loss += loss.item() * targets.size(0)
                correct_val += int((outputs.logits.argmax(dim=-1) == targets).sum())
                total_val_graphs += targets.size(0)

        avg_val_loss = total_val_loss / total_val_graphs
        val_acc = correct_val / total_val_graphs

        print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

        writer.add_scalar("Loss/Train", avg_train_loss, epoch)
        writer.add_scalar("Loss/Validation", avg_val_loss, epoch)
        writer.add_scalar("Accuracy/Train", train_acc, epoch)
        writer.add_scalar("Accuracy/Validation", val_acc, epoch)

        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print(f"== Early stopping triggered at epoch {epoch}. Restoring weights. ==")
            break

    # Close TensorBoard logging process
    writer.close()

    # Load Best Model Checkpoint for Final Inference Evaluation
    if os.path.exists("best_graphormer_model.pt"):
        model.load_state_dict(torch.load("best_graphormer_model.pt"))

    # Test Set Validation Evaluation Phase
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
      for batch in test_loader:
        targets = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        all_targets.extend(targets.numpy())
    accuracy = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro')
    print("\n================ GRAPHORMER TEST REPORT ================")
    print(f"Overall Accuracy : {accuracy:.4f}")
    print(f"Macro Precision  : {precision:.4f}")
    print(f"Macro Recall     : {recall:.4f}")
    print(f"Macro F1 Score   : {f1:.4f}")
    print("========================================================\n")
    print(classification_report(all_targets, all_preds, target_names=['0: False', '1: True', '2: Unknown'], zero_division=0))

if __name__ == "__main__":
    run_graphormer_pipeline(train_graph_list, train_label_list, test_graph_list, test_label_list, val_graph_list, val_label_list)

Executing Graphormer Pipeline on: cuda

--- Beginning Graphormer Training Loop ---
Epoch 01 | Train Loss: 1.1198 Acc: 0.2639 | Val Loss: 1.1076 Acc: 0.3646
Epoch 02 | Train Loss: 1.1051 Acc: 0.3611 | Val Loss: 1.1075 Acc: 0.4062
EarlyStopping Counter: 1 out of 8
Epoch 03 | Train Loss: 1.0977 Acc: 0.3889 | Val Loss: 1.1221 Acc: 0.4062
EarlyStopping Counter: 2 out of 8
Epoch 04 | Train Loss: 1.1006 Acc: 0.4028 | Val Loss: 1.1520 Acc: 0.3021
EarlyStopping Counter: 3 out of 8
Epoch 05 | Train Loss: 1.0944 Acc: 0.3125 | Val Loss: 1.1512 Acc: 0.3229
EarlyStopping Counter: 4 out of 8
Epoch 06 | Train Loss: 1.0934 Acc: 0.3611 | Val Loss: 1.1532 Acc: 0.2083
EarlyStopping Counter: 5 out of 8
Epoch 07 | Train Loss: 1.0835 Acc: 0.4444 | Val Loss: 1.1536 Acc: 0.3125
EarlyStopping Counter: 6 out of 8
Epoch 08 | Train Loss: 1.0673 Acc: 0.3958 | Val Loss: 1.1870 Acc: 0.2083
EarlyStopping Counter: 7 out of 8
Epoch 09 | Train Loss: 1.0631 Acc: 0.4167 | Val Loss: 1.2144 Acc: 0.3646
EarlyStopping Counter:

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

def extract_graph_features(graphs_list):
    """
    Extracts a vector of statistical and structural features for each NetworkX graph.
    """
    features = []
    for G in graphs_list:
        num_nodes = G.number_of_nodes()
        num_edges = G.number_of_edges()
        density = nx.density(G)

        # Degree statistics
        degrees = [d for _, d in G.degree()]
        mean_degree = np.mean(degrees) if degrees else 0
        max_degree = np.max(degrees) if degrees else 0
        std_degree = np.std(degrees) if degrees else 0

        # Connectivity and clustering
        # Convert to undirected for clustering coefficient safely
        undirected_G = G.to_undirected()
        avg_clustering = nx.average_clustering(undirected_G)
        transitivity = nx.transitivity(undirected_G)

        # Basic categorical counts from node attributes if present
        node_types = [data.get("node_type", "") for _, data in G.nodes(data=True)]
        num_premises = sum(1 for t in node_types if "premise" in str(t))
        num_conclusions = sum(1 for t in node_types if "conclusion" in str(t))
        num_quantifiers = sum(1 for t in node_types if "quantifier" in str(t))
        num_keywords = sum(1 for t in node_types if "keyword" in str(t))
        num_terms = sum(1 for t in node_types if "term" in str(t))

        features.append([
            num_nodes, num_edges, density, mean_degree, max_degree, std_degree,
            avg_clustering, transitivity, num_premises, num_conclusions,
            num_quantifiers, num_keywords, num_terms
        ])

    feature_names = [
        "num_nodes", "num_edges", "density", "mean_degree", "max_degree", "std_degree",
        "avg_clustering", "transitivity", "num_premises", "num_conclusions",
        "num_quantifiers", "num_keywords", "num_terms"
    ]
    return pd.DataFrame(features, columns=feature_names)

# Prepare features and target labels
label_map = {'F': 0, 'T': 1, 'U': 2}
y_train = np.array([label_map[lbl] for lbl in train_label_list])
y_test = np.array([label_map[lbl] for lbl in test_label_list])

X_train = extract_graph_features(train_graph_list)
X_test = extract_graph_features(test_graph_list)

# Train Random Forest baseline
rf_baseline = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight="balanced")
rf_baseline.fit(X_train, y_train)

# Inference and Performance Evaluation
y_pred = rf_baseline.predict(X_test)

print("================ RANDOM FOREST BASELINE REPORT ================")
print(f"Overall Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("===============================================================\n")
print(classification_report(y_test, y_pred, target_names=['0: False', '1: True', '2: Unknown'], zero_division=0))

# Display feature importances
importances = rf_baseline.feature_importances_
indices = np.argsort(importances)[::-1]
print("Top Structural Feature Importances:")
for f in range(X_train.shape[1]):
    print(f"{f + 1:2d}) {X_train.columns[indices[f]]:20s}: {importances[indices[f]]:.4f}")

================ RANDOM FOREST BASELINE REPORT ================
Overall Accuracy : 0.2459

              precision    recall  f1-score   support

    0: False       0.20      0.29      0.24        14
     1: True       0.33      0.28      0.30        25
  2: Unknown       0.20      0.18      0.19        22

    accuracy                           0.25        61
   macro avg       0.24      0.25      0.24        61
weighted avg       0.25      0.25      0.25        61

Top Structural Feature Importances:
 1) avg_clustering      : 0.1493
 2) std_degree          : 0.1155
 3) density             : 0.1140
 4) num_edges           : 0.1081
 5) transitivity        : 0.1052
 6) mean_degree         : 0.1031
 7) max_degree          : 0.0777
 8) num_nodes           : 0.0700
 9) num_terms           : 0.0700
10) num_keywords        : 0.0345
11) num_premises        : 0.0307
12) num_quantifiers     : 0.0221
13) num_conclusions     : 0.0000


In [16]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

def extract_semantic_corpus(graphs_list):
    """
    Aggregates all node labels inside each graph into a single string corpus.
    """
    corpus = []
    for G in graphs_list:
        labels = [str(data.get("label", "")) for _, data in G.nodes(data=True)]
        corpus.append(" ".join(labels))
    return corpus

# 1. Create text corpuses for vectorization
train_corpus = extract_semantic_corpus(train_graph_list)
test_corpus = extract_semantic_corpus(test_graph_list)

# 2. Extract TF-IDF features
vectorizer = TfidfVectorizer(max_features=50, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_corpus).toarray()
X_test_tfidf = vectorizer.transform(test_corpus).toarray()

# 3. Build structural features
X_train_struct = extract_graph_features(train_graph_list).values
X_test_struct = extract_graph_features(test_graph_list).values

# 4. Concatenate structural features and semantic TF-IDF embeddings
X_train_combined = np.hstack((X_train_struct, X_train_tfidf))
X_test_combined = np.hstack((X_test_struct, X_test_tfidf))

# 5. Train the Random Forest Baseline on the combined datasets
rf_semantic_baseline = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight="balanced")
rf_semantic_baseline.fit(X_train_combined, y_train)

# 6. Evaluation
y_pred_semantic = rf_semantic_baseline.predict(X_test_combined)

print("================ RANDOM FOREST WITH SEMANTICS REPORT ================")
print(f"Overall Accuracy : {accuracy_score(y_test, y_pred_semantic):.4f}")
print("=====================================================================\n")
print(classification_report(y_test, y_pred_semantic, target_names=['0: False', '1: True', '2: Unknown'], zero_division=0))

================ RANDOM FOREST WITH SEMANTICS REPORT ================
Overall Accuracy : 0.2131

              precision    recall  f1-score   support

    0: False       0.12      0.14      0.13        14
     1: True       0.26      0.28      0.27        25
  2: Unknown       0.22      0.18      0.20        22

    accuracy                           0.21        61
   macro avg       0.20      0.20      0.20        61
weighted avg       0.22      0.21      0.21        61



In [17]:
!pip install -q node2vec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 137.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 93.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.4.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.

In [18]:
import numpy as np
from node2vec import Node2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings

def extract_node2vec_embeddings(graphs_list, dimensions=32, walk_length=10, num_walks=40, workers=1):
    """
    Generates graph-level embedding vectors by averaging node-level Node2Vec embeddings.
    """
    graph_embeddings = []
    # Ignore warnings from Node2Vec random walk configurations
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for i, G in enumerate(graphs_list):
            # Convert to undirected safely as standard Node2Vec requires connected/walkable components
            undirected_G = G.to_undirected()
            if undirected_G.number_of_nodes() == 0:
                graph_embeddings.append(np.zeros(dimensions))
                continue

            # Fit Node2Vec representation mapper on the individual graph
            n2v = Node2Vec(
                undirected_G,
                dimensions=dimensions,
                walk_length=walk_length,
                num_walks=num_walks,
                workers=workers,
                quiet=True
            )
            model = n2v.fit(window=5, min_count=1, batch_words=4)

            # Pool node vectors via spatial mean
            node_vectors = [model.wv[str(node)] for node in undirected_G.nodes() if str(node) in model.wv]
            if node_vectors:
                graph_emb = np.mean(node_vectors, axis=0)
            else:
                graph_emb = np.zeros(dimensions)
            graph_embeddings.append(graph_emb)

    return np.array(graph_embeddings)

print("Generating Node2Vec embeddings for Train set...")
X_train_n2v = extract_node2vec_embeddings(train_graph_list, dimensions=32)
print("Generating Node2Vec embeddings for Test set...")
X_test_n2v = extract_node2vec_embeddings(test_graph_list, dimensions=32)

# Train Random Forest on node2vec features
rf_n2v = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight="balanced")
rf_n2v.fit(X_train_n2v, y_train)

y_pred_n2v = rf_n2v.predict(X_test_n2v)

print("\n================ RANDOM FOREST WITH NODE2VEC REPORT ================")
print(f"Overall Accuracy : {accuracy_score(y_test, y_pred_n2v):.4f}")
print("====================================================================\n")
print(classification_report(y_test, y_pred_n2v, target_names=['0: False', '1: True', '2: Unknown'], zero_division=0))

Generating Node2Vec embeddings for Train set...
Generating Node2Vec embeddings for Test set...

================ RANDOM FOREST WITH NODE2VEC REPORT ================
Overall Accuracy : 0.3279

              precision    recall  f1-score   support

    0: False       0.29      0.14      0.19        14
     1: True       0.36      0.48      0.41        25
  2: Unknown       0.29      0.27      0.28        22

    accuracy                           0.33        61
   macro avg       0.31      0.30      0.29        61
weighted avg       0.32      0.33      0.31        61



In [9]:
# Diagnostics cell to find the source of NaN losses
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Re-create a small test setup
label_map = {'F': 0, 'T': 1, 'U': 2}
train_labels = [label_map[lbl] for lbl in train_label_list]
train_dataset = GraphormerSyllogismDataset(graphs_list=train_graph_list, labels_list=train_labels, max_nodes=32)
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=graphormer_collate_fn)

# Get a single batch
batch = next(iter(train_loader))
targets = batch.pop("labels").to(device)
batch_device = {k: v.to(device) for k, v in batch.items()}

# Let's inspect the shapes and value ranges
print("--- Tensor Diagnostics inside a Batch ---")
for k, v in batch_device.items():
    print(f"{k:20s} | Shape: {list(v.shape)} | Min: {v.min().item()} | Max: {v.max().item()} | Any NaN: {torch.isnan(v).any().item()}")

# Run a single forward pass with the model
config = GraphormerConfig(
    num_classes=3, num_atoms=6, num_edges=2, spatial_pos_max=64,
    num_attention_heads=4, embedding_dim=64, ffn_embedding_dim=64, num_trans_layers=3
)
model = GraphormerForGraphClassification(config).to(device)

# Evaluate the embeddings and model output
with torch.no_grad():
    outputs = model(**batch_device)
    print("\n--- Model Outputs ---")
    print(f"Logits Shape: {outputs.logits.shape}")
    print(f"Logits: {outputs.logits}")
    print(f"Logits Any NaN: {torch.isnan(outputs.logits).any().item()}")


--- Tensor Diagnostics inside a Batch ---
input_nodes          | Shape: [4, 30, 1] | Min: 0 | Max: 5 | Any NaN: False
edge_input           | Shape: [4, 30, 30, 1] | Min: 0 | Max: 1 | Any NaN: False
spatial_pos          | Shape: [4, 30, 30] | Min: 0 | Max: 32 | Any NaN: False
in_degree            | Shape: [4, 30] | Min: 0 | Max: 17 | Any NaN: False
out_degree           | Shape: [4, 30] | Min: 0 | Max: 19 | Any NaN: False
attn_bias            | Shape: [4, 31, 31] | Min: -inf | Max: 0.0 | Any NaN: False
input_edges          | Shape: [4, 30, 30, 1, 1] | Min: 0 | Max: 0 | Any NaN: False
attn_edge_type       | Shape: [4, 30, 30, 1] | Min: 0 | Max: 0 | Any NaN: False

--- Model Outputs ---
Logits Shape: torch.Size([4, 3])
Logits: tensor([[   nan,    nan,    nan],
        [   nan,    nan,    nan],
        [   nan,    nan,    nan],
        [0.0480, 0.0536, 0.1613]], device='cuda:0')
Logits Any NaN: True
